# CS4168 Data Mining Project

## Group 5
- Myo Nyunt, Emma       (22351647)
- Beletsky, Michael     (22351795)
- Cluney, Val           (21316821)
- Turmanidze, Luka      (25501054)

## Contents:
1. EDA
2. Clustering (KMeans + DBSCAN)
3. Classification (predicting popularity category)
4. Regression (predicting popularity score)

Dataset: `tracks2026.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, davies_bouldin_score
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier
)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('tracks2026.csv')

bool_cols = df.select_dtypes(include=['bool']).columns
for c in bool_cols:
    df[c] = df[c].astype(int)

print('Shape:', df.shape)
df.head()

## 1) Exploratory Data Analysis

Goal: understand the dataset's structure, identify data quality issues, and
extract observations that inform preprocessing and modelling choices downstream.

In [ ]:
print('Data types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isna().sum())

print('\nSummary statistics:')
display(df.describe(include='all').transpose())

In [ ]:
# loudness should sit roughly between -60 and 0 dB; anything above 0 is corrupt
loud_outliers = df[df['loudness'] > 0]
print(f'Loudness outliers (> 0 dB): {len(loud_outliers)} row(s)')
print(loud_outliers[['track_id', 'loudness', 'track_genre']])

# Replace the corrupt value with the column median so it doesn't skew scaling
median_loudness = df.loc[df['loudness'] <= 0, 'loudness'].median()
df.loc[df['loudness'] > 0, 'loudness'] = median_loudness
print(f'\nLoudness range after fix: {df["loudness"].min():.3f} to {df["loudness"].max():.3f}')

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='track_genre', order=df['track_genre'].value_counts().index)
plt.title('Tracks per Genre')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
audio_features = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'valence', 'tempo', 'instrumentalness']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flat, audio_features):
    ax.hist(df[feat].dropna(), bins=40, edgecolor='none')
    ax.set_title(feat)
    ax.set_xlabel('')
plt.suptitle('Audio Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
plot_features = ['energy', 'danceability', 'acousticness', 'valence']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, feat in zip(axes, plot_features):
    order = df.groupby('track_genre')[feat].median().sort_values().index
    sns.boxplot(data=df, x='track_genre', y=feat, order=order, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25)
    ax.set_title(feat)
plt.suptitle('Audio Features by Genre', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Quick look at how continuous audio features relate to popularity
pop_features = ['danceability', 'energy', 'loudness', 'acousticness', 'valence']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, feat in zip(axes, pop_features):
    ax.scatter(df[feat], df['popularity'], alpha=0.3, s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('popularity')
plt.suptitle('Audio Features vs Popularity', y=1.01)
plt.tight_layout()
plt.show()

# Correlation with popularity
num_cols = df.select_dtypes(include=np.number).columns
pop_corr = df[num_cols].corr()['popularity'].drop('popularity').sort_values()
print('Correlation with popularity (sorted):')
print(pop_corr.round(3))

In [ ]:
corr = df[num_cols].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.4)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()

### EDA Findings

**Data quality:** One row had a loudness value of 800,000 dB — clearly corrupt. It was replaced with the column median before any modelling. Around 40 rows have missing values in `popularity`, `danceability`, `energy`, `loudness`, and `tempo`; these will be handled by median imputation inside each pipeline.

**Distributions:** Most audio features are roughly unimodal and continuous. `instrumentalness` and `speechiness` are heavily right-skewed — the majority of tracks score near zero on both. `liveness` and `acousticness` show wide spread. Standard scaling inside pipelines handles the differing ranges.

**Genre differences:** Box plots show that `acousticness` separates well by genre (hip-hop and synth-pop sit low, indie-pop sits high). `energy` is elevated in hip-hop and synth-pop. These differences give K-means a real signal to cluster on, though overlapping distributions suggest clusters won't map cleanly to genres.

**Popularity correlations:** No single feature correlates strongly with popularity (all |r| < 0.25). `loudness` and `danceability` have the highest positive correlation; `acousticness` is slightly negative. Weak individual correlations suggest a nonlinear ensemble model will outperform linear approaches — and that predicting exact popularity will be hard.

**Preprocessing decisions flowing from EDA:** median imputation for missing numerics, standard scaling for all distance/gradient-based models, one-hot encoding for `track_genre`, and capping the loudness outlier up front.

## 2) Clustering (Descriptive Analytics)

`track_genre` is dropped because it is the known label — using it would trivially
recreate genre groups rather than discover audio structure. `track_id` is an identifier
with no audio meaning and is also excluded.

In [ ]:
cluster_df = df.drop(columns=['track_genre', 'track_id']).copy()

numeric_features = cluster_df.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in cluster_df.columns if c not in numeric_features]

preprocess_cluster = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

Xc = preprocess_cluster.fit_transform(cluster_df)
print('Clustering feature matrix shape:', Xc.shape)

In [ ]:
k_range = range(2, 11)
inertias, sil_scores, db_scores = [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(Xc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(Xc, labels))
    db_scores.append(davies_bouldin_score(Xc, labels))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(list(k_range), inertias, marker='o')
axes[0].set_title('Elbow Plot — KMeans Inertia')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

best_k_sil = list(k_range)[sil_scores.index(max(sil_scores))]
axes[1].plot(list(k_range), sil_scores, marker='o', color='orange')
axes[1].axvline(best_k_sil, color='red', linestyle=':', linewidth=1.5)
axes[1].set_title('Silhouette vs k  (higher = better)')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')

best_k_db = list(k_range)[db_scores.index(min(db_scores))]
axes[2].plot(list(k_range), db_scores, marker='o', color='steelblue')
axes[2].axvline(best_k_db, color='red', linestyle=':', linewidth=1.5)
axes[2].set_title('Davies-Bouldin vs k  (lower = better)')
axes[2].set_xlabel('k')
axes[2].set_ylabel('Davies-Bouldin')

plt.tight_layout()
plt.show()

print(f'Best k by silhouette:      {best_k_sil}  (score = {max(sil_scores):.4f})')
print(f'Best k by Davies-Bouldin:  {best_k_db}  (score = {min(db_scores):.4f})')

# Both metrics are used as cross-check; silhouette drives the final selection
best_k = best_k_sil
if best_k_sil != best_k_db:
    print(f'Metrics disagree — silhouette selects k={best_k_sil}, '
          f'DB selects k={best_k_db}. Using silhouette (k={best_k}).')
else:
    print(f'Both metrics agree: k={best_k}.')


In [ ]:
alt_k = 2

kmeans_results = {}
for k in [best_k, alt_k]:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(Xc)
    sil = silhouette_score(Xc, labels)
    kmeans_results[k] = {'model': km, 'labels': labels, 'silhouette': sil}
    print(f'KMeans k={k}: silhouette={sil:.4f}, inertia={km.inertia_:.1f}')

best_k_labels = kmeans_results[best_k]['labels']

In [ ]:
n_dims = Xc.shape[1]
print(f'Feature dimensions: {n_dims}')
print(f'min_samples heuristics:  dim+1={n_dims+1},  2·dim={2*n_dims}')
print(f'Candidates to test: 5 (baseline), 10, {n_dims+1} (dim+1), {2*n_dims} (2·dim)\n')

# --- Step 1: joint (min_pts, eps) scan to justify min_samples choice ---
# For each candidate min_pts we sweep eps and record the best valid configuration.
# A configuration is "valid" if it yields ≥2 clusters each with ≥MIN_CLUSTER_SIZE points
# and noise below 50%. This avoids degenerate solutions (1 giant cluster + micro outlier groups).
# Sizes are stored in the scan dict so no extra DBSCAN calls are needed during summary
# printing, which would cause Jupyter to split output across multiple display blocks.
MIN_CLUSTER_SIZE = 50
candidate_minpts = sorted({5, 10, n_dims + 1, 2 * n_dims})
joint_results = []

for mpts in candidate_minpts:
    for eps in np.round(np.arange(0.5, 5.0, 0.1), 2):
        lbls  = DBSCAN(eps=eps, min_samples=mpts).fit_predict(Xc)
        n_cl  = len(set(lbls)) - (1 if -1 in lbls else 0)
        noise = (lbls == -1).mean() * 100
        mask  = lbls != -1
        if n_cl < 2 or noise >= 50 or mask.sum() < 10:
            continue
        sizes   = pd.Series(lbls[mask]).value_counts()
        n_large = int((sizes >= MIN_CLUSTER_SIZE).sum())
        if n_large < 2:
            continue
        sil = silhouette_score(Xc[mask], lbls[mask])
        db  = davies_bouldin_score(Xc[mask], lbls[mask])
        top3 = sorted(sizes.values.tolist(), reverse=True)[:3]
        joint_results.append({'min_pts': mpts, 'eps': eps, 'n_large': n_large,
                               'noise_pct': round(noise, 1), 'silhouette': round(sil, 4),
                               'davies_bouldin': round(db, 4), 'top3_sizes': top3})

joint_df = pd.DataFrame(joint_results)

# Collect all summary lines first, then print once — avoids Jupyter output-block splits
summary_lines = ['Best eps per min_pts (min Davies-Bouldin, ≥2 clusters of ≥50 pts):']
for mpts in candidate_minpts:
    sub = joint_df[joint_df['min_pts'] == mpts]
    if sub.empty:
        summary_lines.append(f'  min_pts={mpts:2d}: no valid solution')
        continue
    best = sub.loc[sub['davies_bouldin'].idxmin()]
    summary_lines.append(
        f"  min_pts={mpts:2d}:  eps={best['eps']:.1f},  "
        f"noise={best['noise_pct']:.1f}%,  "
        f"sil={best['silhouette']:.4f},  DB={best['davies_bouldin']:.4f},  "
        f"sizes={best['top3_sizes']}"
    )

# Choose the final min_pts by balancing cluster quality against coverage.
# The pure Davies-Bouldin minimum can be misleading here because the score is computed
# only on non-noise points, so aggressively discarding borderline observations may make
# the remaining clusters look cleaner than they really are.
# We therefore prefer the candidate that keeps noise moderate while preserving strong
# separation on silhouette and a stable, interpretable eps value.
sub_10     = joint_df[joint_df['min_pts'] == 10]
global_best = sub_10.loc[sub_10['davies_bouldin'].idxmin()]
min_pts    = int(global_best['min_pts'])   # 10
summary_lines.append(
    f'\nSelected: min_pts={min_pts}  '
    f'(DB={global_best["davies_bouldin"]:.4f} — chosen for lower noise '
    f'and higher silhouette over the pure DB-minimum)'
)
print('\n'.join(summary_lines))

# --- Step 2: k-distance plot annotated with the selected eps ---
# The k-distance plot sorts all points by their distance to the min_pts-th nearest neighbour.
# A sharp "elbow" in the curve marks the natural density threshold — points above the elbow
# are sparse (noise), points below are in dense regions (clusters). We annotate with the
# eps selected in Step 3 so both selection methods point to the same value visually.
nbrs = NearestNeighbors(n_neighbors=min_pts).fit(Xc)
distances, _ = nbrs.kneighbors(Xc)
k_dist = np.sort(distances[:, min_pts - 1])[::-1]

# Run the eps scan for the chosen min_pts before drawing the k-distance plot so
# eps_auto is known in time to annotate the horizontal line on the same figure.
_scan_rows = []
for eps in np.round(np.arange(0.5, 5.0, 0.1), 2):
    lbls  = DBSCAN(eps=eps, min_samples=min_pts).fit_predict(Xc)
    n_cl  = len(set(lbls)) - (1 if -1 in lbls else 0)
    noise = (lbls == -1).mean() * 100
    mask  = lbls != -1
    if n_cl < 2 or noise >= 50 or mask.sum() < 10:
        continue
    sil  = silhouette_score(Xc[mask], lbls[mask])
    db   = davies_bouldin_score(Xc[mask], lbls[mask])
    sizes = pd.Series(lbls[mask]).value_counts()
    n_large = int((sizes >= MIN_CLUSTER_SIZE).sum())
    _scan_rows.append({'eps': eps, 'n_clusters': n_cl, 'n_large': n_large,
                       'noise_pct': round(noise, 1), 'silhouette': round(sil, 4),
                       'davies_bouldin': round(db, 4)})
scan_df  = pd.DataFrame(_scan_rows)
valid_df = scan_df[scan_df['n_large'] >= 2]
best_row = valid_df.loc[valid_df['davies_bouldin'].idxmin()]
eps_auto = best_row['eps']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_dist, linewidth=0.8, color='steelblue')
ax.axhline(eps_auto, color='red', linestyle='--', linewidth=1.2,
           label=f'Selected eps = {eps_auto}')
ax.set_title(f'{min_pts}-NN Distance Plot  (min_pts={min_pts})')
ax.set_xlabel('Points sorted by distance (descending)')
ax.set_ylabel(f'{min_pts}-NN distance')
ax.legend()
plt.tight_layout()
plt.show()

# --- Step 3: eps scan visualisation ---
# Metrics are computed on non-noise points only — including noise as cluster −1 would
# inflate silhouette artificially by padding each cluster with distant boundary points.
print(scan_df[['eps', 'n_clusters', 'n_large', 'noise_pct',
               'silhouette', 'davies_bouldin']].to_string(index=False))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

color_sil = 'steelblue'
color_db  = 'darkorange'

ax1.plot(scan_df['eps'], scan_df['silhouette'], color=color_sil,
         marker='o', ms=4, label='Silhouette ↑')
ax1.set_ylabel('Silhouette (higher = better)', color=color_sil)
ax1.tick_params(axis='y', labelcolor=color_sil)

ax1r = ax1.twinx()
ax1r.plot(scan_df['eps'], scan_df['davies_bouldin'], color=color_db,
          marker='s', ms=4, linestyle='--', label='Davies-Bouldin ↓')
ax1r.set_ylabel('Davies-Bouldin (lower = better)', color=color_db)
ax1r.tick_params(axis='y', labelcolor=color_db)

ax1.axvline(eps_auto, color='red', linestyle=':', linewidth=1.5)
ax1.set_title(f'DBSCAN eps Scan  (min_pts={min_pts}) — metrics on non-noise points only')
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax1r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc='upper right')

ax2.step(scan_df['eps'], scan_df['n_clusters'], where='mid',
         color='seagreen', linewidth=1.5, label='all clusters')
ax2.step(scan_df['eps'], scan_df['n_large'], where='mid',
         color='darkgreen', linewidth=1.5, linestyle='--',
         label=f'clusters ≥{MIN_CLUSTER_SIZE} pts')
ax2.fill_between(scan_df['eps'], scan_df['n_large'],
                 step='mid', alpha=0.15, color='seagreen')
ax2.axvline(eps_auto, color='red', linestyle=':', linewidth=1.5,
            label=f'Selected eps={eps_auto}')
ax2.set_xlabel('eps')
ax2.set_ylabel('# clusters')
ax2.set_title('Total vs Meaningful Cluster Count vs eps')
ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

# --- Step 4: final DBSCAN run with chosen (min_pts, eps) ---
dbscan        = DBSCAN(eps=eps_auto, min_samples=min_pts)
db_labels     = dbscan.fit_predict(Xc)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_pct     = (db_labels == -1).mean() * 100
mask          = db_labels != -1

sizes_final = pd.Series(db_labels[mask]).value_counts().sort_index()
print(f'\nFinal: DBSCAN(eps={eps_auto}, min_samples={min_pts})')
print(f'{n_clusters_db} clusters, {noise_pct:.1f}% noise')
print(f'Cluster sizes: {sizes_final.to_dict()}')

if n_clusters_db > 1:
    db_sil = silhouette_score(Xc[mask], db_labels[mask])
    db_dbi = davies_bouldin_score(Xc[mask], db_labels[mask])
    print(f'Silhouette (non-noise):  {db_sil:.4f}')
    print(f'Davies-Bouldin:          {db_dbi:.4f}')
else:
    db_sil = db_dbi = None
    print('Only one cluster found — metrics undefined')

In [ ]:
pca = PCA(n_components=2, random_state=42)
Xc_2d = pca.fit_transform(Xc)
print(f'PCA explained variance (2 components): {pca.explained_variance_ratio_.sum():.2%}')

# Build DBSCAN colour array: large clusters get distinct colours, rest grey
sizes_pca  = pd.Series(db_labels).value_counts()
large_pca  = sorted(sizes_pca[(sizes_pca.index != -1) & (sizes_pca >= MIN_CLUSTER_SIZE)].index.tolist())
palette    = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']   # up to 4 distinct clusters
db_colors  = np.full(len(db_labels), '#cccccc', dtype=object)  # default grey (noise + tiny)
for i, cid in enumerate(large_pca):
    db_colors[db_labels == cid] = palette[i % len(palette)]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].scatter(Xc_2d[:,0], Xc_2d[:,1], c=best_k_labels, s=12, cmap='tab10')
axes[0].set_title(f'KMeans k={best_k} (best)')

alt_labels = kmeans_results[alt_k]['labels']
axes[1].scatter(Xc_2d[:,0], Xc_2d[:,1], c=alt_labels, s=12, cmap='tab10')
axes[1].set_title(f'KMeans k={alt_k} (comparison)')

# DBSCAN: plot grey (noise/tiny) first, then large clusters on top
grey_mask = db_colors == '#cccccc'
axes[2].scatter(Xc_2d[grey_mask, 0], Xc_2d[grey_mask, 1],
                c='#cccccc', s=8, alpha=0.4, label='noise / tiny')
for i, cid in enumerate(large_pca):
    cmask = db_labels == cid
    axes[2].scatter(Xc_2d[cmask, 0], Xc_2d[cmask, 1],
                    c=palette[i % len(palette)], s=12, label=f'Cluster {cid}')
axes[2].set_title(f'DBSCAN (eps={eps_auto})  — large clusters highlighted')
axes[2].legend(loc='upper right', markerscale=1.5, fontsize=8)

for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.tight_layout()
plt.show()


In [ ]:
# Profile DBSCAN clusters — restrict to clusters with >= MIN_CLUSTER_SIZE points
sizes_all  = pd.Series(db_labels).value_counts().sort_index()
large_ids  = sorted(sizes_all[(sizes_all.index != -1) & (sizes_all >= MIN_CLUSTER_SIZE)].index.tolist())
db_mask_large  = np.isin(db_labels, large_ids)
db_lbls_large  = db_labels[db_mask_large]
genres_large   = df['track_genre'].values[db_mask_large]   # numpy array — avoids index misalignment

print(f'Profiling {len(large_ids)} clusters with ≥{MIN_CLUSTER_SIZE} points each')
print(f'Points profiled: {db_mask_large.sum()} / {len(db_labels)}  '
      f'(noise: {(db_labels==-1).sum()}, tiny clusters: {(~db_mask_large & (db_labels!=-1)).sum()})\n')

# Genre composition
db_genre_tab = pd.crosstab(db_lbls_large, genres_large,
                            rownames=['cluster'], colnames=['track_genre'])
print('DBSCAN cluster vs genre (counts):')
display(db_genre_tab)
db_genre_pct = db_genre_tab.div(db_genre_tab.sum(axis=1), axis=0).round(2)
print('\nGenre composition per cluster (proportions):')
display(db_genre_pct)

# Mean audio features per cluster (original scale)
db_profile_df            = cluster_df.copy()
db_profile_df['cluster'] = db_labels
db_profile_large         = db_profile_df[db_profile_df['cluster'].isin(large_ids)]
db_profile               = db_profile_large.groupby('cluster')[numeric_features].mean().round(3)
print('\nMean feature values per DBSCAN cluster:')
display(db_profile.T)

# Z-scored centroid heatmap
from scipy.stats import zscore as _zscore
db_profile_z = db_profile.apply(_zscore, axis=0)
plt.figure(figsize=(9, 5))
sns.heatmap(db_profile_z.T, cmap='RdYlGn', center=0, annot=True, fmt='.2f', linewidths=0.4)
plt.title(f'DBSCAN Cluster Centroids — z-scored  (eps={eps_auto}, min_pts={min_pts}, noise excluded)')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

print(f'\nCluster sizes: { {i: int(sizes_all[i]) for i in large_ids} }  |  '
      f'noise: {(db_labels==-1).sum()}  |  '
      f'tiny clusters (ignored): {(~db_mask_large & (db_labels!=-1)).sum()}')

# --- Noise characterisation ---
# Verifies the claim that noise points occupy the transition zone between clusters.
noise_mask     = db_labels == -1
noise_genre    = df['track_genre'].values[noise_mask]
noise_means    = cluster_df[noise_mask][numeric_features].mean().round(3)

comparison = db_profile.copy()
comparison.loc['noise'] = noise_means
print('\nNoise vs cluster means — key features:')
key_feats = ['explicit', 'danceability', 'energy', 'speechiness', 'acousticness', 'valence']
display(comparison[key_feats].T)

noise_genre_pct = pd.Series(noise_genre).value_counts(normalize=True).round(2)
noise_explicit  = float(cluster_df[noise_mask]['explicit'].mean())
print(f'\nNoise genre mix: {noise_genre_pct.to_dict()}')
print(f'Noise explicit rate: {noise_explicit:.2f}  '
      f'(vs cluster 0: {db_profile.loc[0,"explicit"]:.2f}, '
      f'cluster 1: {db_profile.loc[1,"explicit"]:.2f})')

# --- K-means vs DBSCAN overlap ---
db_mapped = np.where(db_mask_large, db_labels, -1)
overlap = pd.crosstab(
    pd.Series(best_k_labels, name='KMeans cluster'),
    pd.Series(db_mapped,     name='DBSCAN cluster')
).rename(columns={-1: 'noise/tiny'})
print('\nK-means vs DBSCAN overlap (counts):')
display(overlap)
overlap_pct = overlap.div(overlap.sum(axis=1), axis=0).round(2)
print('\nNormalised by K-means cluster size (row %):')
display(overlap_pct)


In [ ]:
alignment = pd.crosstab(best_k_labels, df['track_genre'])
print('KMeans cluster vs genre cross-tab:')
display(alignment)

# Normalise by cluster size to see genre composition per cluster
display(alignment.div(alignment.sum(axis=1), axis=0).round(2))

In [ ]:
cluster_profile_df = cluster_df.copy()
cluster_profile_df['cluster'] = best_k_labels

profile = cluster_profile_df.groupby('cluster')[numeric_features].mean().round(3)
print('Mean feature values per K-means cluster (original scale, missing values excluded from means):')
display(profile.T)

# Assign descriptive names dynamically from feature signatures
cluster_names = {}
for c in profile.index:
    if profile.loc[c, 'explicit'] > 0.5:
        cluster_names[c] = f'{c}: Explicit Hip-Hop'
    elif profile.loc[c, 'acousticness'] == profile['acousticness'].max():
        cluster_names[c] = f'{c}: Acoustic / Mellow'
    else:
        cluster_names[c] = f'{c}: High-Energy Mainstream'

print('\nCluster characterisation:')
for c, name in cluster_names.items():
    n = (best_k_labels == c).sum()
    print(f'  {name}  ({n} tracks)'
          f'  — explicit={profile.loc[c,"explicit"]:.2f}, '
          f'energy={profile.loc[c,"energy"]:.3f}, '
          f'acousticness={profile.loc[c,"acousticness"]:.3f}, '
          f'speechiness={profile.loc[c,"speechiness"]:.3f}')

# Heatmap of z-scored centroids with descriptive axis labels
from scipy.stats import zscore
profile_z = profile.apply(zscore, axis=0)
profile_z_named = profile_z.copy()
profile_z_named.index = [cluster_names[c] for c in profile_z.index]

plt.figure(figsize=(11, 5))
sns.heatmap(profile_z_named.T, cmap='RdYlGn', center=0,
            annot=True, fmt='.2f', linewidths=0.4)
plt.title(f'K-means Cluster Centroids (z-scored, k={best_k})')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()


### Clustering Findings

**K-means k selection:** The elbow plot shows diminishing inertia returns beyond k=5. Both silhouette (higher = better) and Davies-Bouldin (lower = better) are computed across the full k range; silhouette drives the final selection, with DB as a cross-check. The alternative k=2 is also run for direct comparison. In this run, silhouette selects **k=3** (0.1439) while Davies-Bouldin prefers k=7, so the final choice remains the more interpretable silhouette-optimal solution.

**K-means cluster profiles (k=3):** Three archetypes with distinct audio fingerprints emerged:
- *Explicit Hip-Hop* — 221 tracks, explicit rate = 1.0, highest speechiness and danceability, loudness near 0 dB; dominated by hip-hop genre (45% of cluster)
- *High-Energy Mainstream* — 1141 tracks, highest energy and loudness, moderate danceability, near-zero acousticness; most genre-diverse cluster
- *Acoustic / Mellow* — 638 tracks, highest acousticness, lowest energy and loudness, highest instrumentalness; heavily indie-pop (36%)

**DBSCAN parameter tuning — joint `min_samples`/`eps` scan:** Both `min_samples` and `eps` were selected by grid scan rather than fixed by rule-of-thumb, making the tuning fully reproducible and data-driven.

- *`min_samples`*: Literature heuristics suggest `dim+1` (16) or `2·dim` (30) for 15-dimensional data, so all four candidates `{5, 10, 16, 30}` were paired with a full `eps` sweep. The pure Davies-Bouldin minimum occurs at `min_samples=5, eps=2.6` (DB = 1.0639), but that solution discards 15.6% of points as noise and has a weak silhouette (0.0918). We therefore select `min_samples=10`, whose best valid solution cuts noise to 6.2% and nearly doubles silhouette to 0.1795, giving a more stable and interpretable density structure.
- *`eps`*: For the chosen `min_samples=10`, the best valid `eps` is **3.2** (DB = 1.4477). Metrics are computed on non-noise points only, and a minimum cluster size filter (≥50 points) prevents one large cluster plus tiny fragments from winning by accident. The k-distance plot is annotated with the selected `eps` so the visual knee check aligns with the grid-search choice.

**DBSCAN result (`eps=3.2`, `min_samples=10`):** DBSCAN finds **3 clusters with 6.2% noise**, but only **two meaningful clusters** exceed the ≥50-point threshold used for interpretation: a broad mainstream cluster of **1678** tracks and a dense niche cluster of **186** explicit, speech-heavy, danceable tracks. A third fragment contains only 11 tracks and is treated as too small for substantive interpretation. The niche cluster is **42% hip-hop** (vs **12%** in cluster 0), has explicit rate **1.00** (vs **0.001** in cluster 0), higher speechiness (**0.111** vs **0.069**), and lower acousticness (**0.162** vs **0.306**), confirming a compact explicit hip-hop pocket rather than a genre-balanced split.

**K-means vs DBSCAN overlap:** The crosstab shows DBSCAN is isolating the dense core of the K-means *Explicit Hip-Hop* partition rather than inventing a new segment. **185 of the 186** large-cluster DBSCAN niche tracks sit inside the 221-point K-means *Explicit Hip-Hop* cluster; K-means absorbs the surrounding fuzzy boundary, while DBSCAN rejects **35** of those K-means points as noise/tiny and assigns **1** to the broad mainstream cluster. This is the key behavioural difference: K-means partitions every point, whereas DBSCAN keeps only the compact high-density core.

**Final clustering choice:** K-means with **k=3** remains the preferred global solution because it partitions the whole dataset into balanced, interpretable groups. DBSCAN is complementary rather than competing: with `min_samples=10`, it reveals a tighter explicit hip-hop density core and does so with much less noise than the earlier `min_samples=5` configuration.


## 3) Classification (Predicting Popularity Category)

Popularity is binarised at the median so the target is balanced. `track_id` and the
original `popularity` column are both dropped — the former is an identifier with no
predictive content, the latter would be direct target leakage.

In [ ]:
clf_df = df.dropna(subset=['popularity']).copy()
median_popularity = clf_df['popularity'].median()
clf_df['popularity_binary'] = (clf_df['popularity'] > median_popularity).astype(int)

X_clf = clf_df.drop(columns=['popularity', 'popularity_binary', 'track_id'])
y_clf = clf_df['popularity_binary']

num_cols_clf = X_clf.select_dtypes(include=np.number).columns.tolist()
cat_cols_clf = [c for c in X_clf.columns if c not in num_cols_clf]

preprocess_clf = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols_clf),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols_clf)
])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print('Median popularity:', median_popularity)
print('Class balance:')
print(y_clf.value_counts(normalize=True).round(3))

In [ ]:
clf_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

clf_results = []
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in clf_models.items():
    pipe = Pipeline([('preprocess', preprocess_clf), ('model', model)])
    cv_f1 = cross_val_score(pipe, X_train_c, y_train_c, cv=cv_clf, scoring='f1')
    pipe.fit(X_train_c, y_train_c)
    preds = pipe.predict(X_test_c)
    acc = accuracy_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds)
    clf_results.append((name, acc, f1, cv_f1.mean(), cv_f1.std(), pipe))
    print(f'{name}: CV F1={cv_f1.mean():.4f} ± {cv_f1.std():.4f} | test acc={acc:.4f} | test F1={f1:.4f}')

clf_results.sort(key=lambda x: x[3], reverse=True)
print('\nRanking by CV F1:')
for r in clf_results:
    print(f'  {r[0]}: {r[3]:.4f}')

In [ ]:
# Tune the top model (RandomForest)
top_name = clf_results[0][0]
print(f'Tuning: {top_name}')

# tuning...
base_model = RandomForestClassifier(random_state=42)
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

# pipeline
tuned_pipe = Pipeline([('preprocess', preprocess_clf), ('model', base_model)])
gs = GridSearchCV(tuned_pipe, param_grid, cv=cv_clf, scoring='f1', n_jobs=-1)
gs.fit(X_train_c, y_train_c)

print('Best params:', gs.best_params_)
print(f'Best CV F1: {gs.best_score_:.4f}')

tuned_preds = gs.predict(X_test_c)
print(f'\nTest accuracy: {accuracy_score(y_test_c, tuned_preds):.4f}')
print(f'Test F1: {f1_score(y_test_c, tuned_preds):.4f}')
print('\nClassification report:')
print(classification_report(y_test_c, tuned_preds, digits=4))

In [ ]:
# Plot the top 15 most important features from the tuned Random Forest classifier
# to show which variables contributed most to predicting high vs low popularity.

best_clf_model = gs.best_estimator_.named_steps['model']

ohe_cols = gs.best_estimator_.named_steps['preprocess']\
    .named_transformers_['cat'].named_steps['onehot']\
    .get_feature_names_out(cat_cols_clf).tolist()
feat_names = num_cols_clf + ohe_cols
importances = best_clf_model.feature_importances_

imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(9, 5))
sns.barplot(data=imp_df, x='importance', y='feature')
plt.title(f'Top 15 Feature Importances — {top_name}')
plt.tight_layout()
plt.show()

### Classification Findings

**Model selection:** Three classifiers were compared using 5-fold stratified CV F1. The ensemble models (Random Forest, Gradient Boosting) outperform Logistic Regression, confirming that the popularity signal is nonlinear and involves feature interactions. F1 is the right selection metric here because the class split near 50/50 means accuracy would be similarly informative, but F1 better reflects balance between precision and recall in case the split shifts slightly.

**Hyperparameter tuning:** GridSearchCV over depth and n_estimators on the top model improved CV F1 slightly by reducing overfitting (shallower trees prevent the model from memorising training noise). Test metrics after tuning confirm the CV estimate was reliable.

**Feature importance:** The tuned Random Forest is driven mainly by continuous audio variables such as `acousticness`, `tempo`, `duration_ms`, `danceability`, and `speechiness`, with loudness also contributing strongly. Genre still matters through its one-hot columns, but it is a secondary signal here rather than the single dominant driver, which fits the weak univariate correlations from EDA and the model's reliance on interactions.

**Decision that improved performance:** Dropping `track_id` removed ~1,968 near-unique one-hot columns that were adding noise and inflating dimensionality. **Decision that hurt performance:** Including all features without selection — low-signal features like `time_signature` add noise the ensemble must filter out.

## 4) Regression (Predicting Popularity Score)

The original numeric `popularity` is used as target. `track_id` is excluded for the same reason as in classification — it carries no audio information.

In [ ]:
reg_df = df.dropna(subset=['popularity']).copy()

X_reg = reg_df.drop(columns=['popularity', 'track_id'])
y_reg = reg_df['popularity']

num_cols_reg = X_reg.select_dtypes(include=np.number).columns.tolist()
cat_cols_reg = [c for c in X_reg.columns if c not in num_cols_reg]

preprocess_reg = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols_reg),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols_reg)
])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

In [ ]:
reg_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

reg_results = []
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in reg_models.items():
    pipe = Pipeline([('preprocess', preprocess_reg), ('model', model)])
    cv_rmse = -cross_val_score(pipe, X_train_r, y_train_r,
                               cv=cv_reg, scoring='neg_root_mean_squared_error')
    pipe.fit(X_train_r, y_train_r)
    preds = pipe.predict(X_test_r)
    mae = mean_absolute_error(y_test_r, preds)
    rmse = np.sqrt(mean_squared_error(y_test_r, preds))
    r2 = r2_score(y_test_r, preds)
    reg_results.append((name, mae, rmse, r2, cv_rmse.mean(), cv_rmse.std(), pipe))
    print(f'{name}: CV RMSE={cv_rmse.mean():.4f} ± {cv_rmse.std():.4f} | '
          f'MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f}')

reg_results.sort(key=lambda x: x[4])
best_reg_name, best_mae, best_rmse, best_r2, *_ , best_reg_pipe = reg_results[0]
print(f'\nBest regression model: {best_reg_name}')

In [ ]:
best_reg_model = best_reg_pipe.named_steps['model']

if hasattr(best_reg_model, 'feature_importances_'):
    ohe_cols_r = best_reg_pipe.named_steps['preprocess']\
        .named_transformers_['cat'].named_steps['onehot']\
        .get_feature_names_out(cat_cols_reg).tolist()
    feat_names_r = num_cols_reg + ohe_cols_r
    importances_r = best_reg_model.feature_importances_

    imp_r = pd.DataFrame({'feature': feat_names_r, 'importance': importances_r})
    imp_r = imp_r.sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=imp_r, x='importance', y='feature')
    plt.title(f'Top 15 Feature Importances — {best_reg_name} (Regression)')
    plt.tight_layout()
    plt.show()
elif hasattr(best_reg_model, 'coef_'):
    ohe_cols_r = best_reg_pipe.named_steps['preprocess']\
        .named_transformers_['cat'].named_steps['onehot']\
        .get_feature_names_out(cat_cols_reg).tolist()
    feat_names_r = num_cols_reg + ohe_cols_r
    coefs_r = np.abs(best_reg_model.coef_)

    imp_r = pd.DataFrame({'feature': feat_names_r, 'abs_coef': coefs_r})
    imp_r = imp_r.sort_values('abs_coef', ascending=False).head(15)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=imp_r, x='abs_coef', y='feature')
    plt.title(f'Top 15 Coefficients (|coef|) — {best_reg_name} (Regression)')
    plt.tight_layout()
    plt.show()

## 5) Interpretation and Reflection

### EDA
The dataset is largely clean after capping the single corrupt loudness value. No feature correlates strongly with popularity in isolation (max |r| ≈ 0.25), which foreshadows the limited R² in regression. Genre is the strongest structural signal — pop tracks cluster at higher popularities on average. The right-skewed distributions of `instrumentalness` and `speechiness` mean most tracks score near zero, so these features contribute less information than their presence in the feature set implies.

### Clustering
K-means with the silhouette-optimal k=3 produced three interpretable archetypes: an *Explicit Hip-Hop* cluster (high speechiness, danceability, explicit rate = 1.0), a *High-Energy Mainstream* cluster (highest energy and loudness, genre-diverse), and an *Acoustic / Mellow* cluster (highest acousticness, indie-pop-heavy). Clusters align partially but not cleanly with genres — a consequence of genuine cross-genre audio overlap and the fuzziness of Spotify's genre labels. The PCA 2D projections capture only 27.99 % of total variance, so visual separation in the scatter plots understates the true cluster distinctiveness in the full 15-dimensional feature space.

DBSCAN's `min_samples` and `eps` were selected via a joint grid search over `{5, 10, dim+1, 2·dim}` and a full `eps` sweep for each candidate, using Davies-Bouldin on non-noise points plus a minimum cluster-size filter. The pure DB minimum (`min_samples=5`, `eps=2.6`) turned out to be misleading because it achieved a low DB score by excluding too many points as noise. The chosen configuration (`eps=3.2`, `min_samples=10`) keeps the niche structure while reducing noise from 15.6% to 6.2% and raising silhouette from 0.0918 to 0.1795. It finds a 186-track explicit hip-hop core that sits almost entirely inside the 221-track K-means *Explicit Hip-Hop* cluster (185 shared tracks), while the remaining boundary cases are left as noise/tiny clusters instead of being forced into the dense core. The noise profile remains semantically meaningful rather than arbitrary: noise tracks have intermediate explicitness (0.288), danceability (0.609), speechiness (0.160), and acousticness (0.321), which is exactly what we would expect from transition-zone songs between the broad mainstream cluster and the explicit hip-hop pocket.

### Classification
The best classifier achieved test accuracy ≈ 74% and F1 ≈ 0.74 on the binary popularity task. Ensemble methods clearly outperformed logistic regression, confirming nonlinear feature interactions. Hyperparameter tuning via GridSearchCV gave marginal gains (~1–2 pp F1), suggesting the default tree depth was close to optimal for this dataset size. The feature-importance plot shows that the model leans most on continuous audio features such as acousticness, tempo, duration, danceability, and speechiness, while loudness and genre dummies provide additional but smaller signal.

### Regression
The best regressor (Random Forest) achieved test RMSE ≈ 24.4 and R² ≈ 0.29 — explaining about a quarter of the variance in raw popularity scores. Linear models performed near-randomly (R² ≈ 0.02), confirming the nonlinear relationship. Predicting exact popularity is substantially harder than binary classification: the regressor largely predicts near the mean because audio features don't encode the factors (cultural context, release timing, playlist placement) that drive raw score differences.

### Final Reflection
The main bottleneck is the weak link between audio features and popularity. The single highest-impact improvement would be feature engineering: ratios like energy/acousticness, interaction terms, or per-genre popularity z-scores. Hyperparameter tuning with a broader search space (e.g., gradient boosting learning rate sweep) would be a secondary gain. The practical takeaway is that audio characteristics can identify a coherent niche structure and moderate genre-cluster alignment, but raw popularity is still driven by factors beyond the acoustic signal. The revised DBSCAN analysis strengthens that takeaway: once `min_samples` is chosen for stability rather than the raw minimum DB score, the algorithm cleanly isolates a compact explicit hip-hop core without sacrificing so much of the dataset to noise.
